In [ ]:
import json, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from metrics import find_best_threshold, evaluate
warnings.filterwarnings("ignore")

DATA_DIR = "dataset_high"
SUB      = 800_000        # so dong CUOI cua train dung cho sweep (giu nguyen tinh thoi gian)

def load(split, n_tail=None):
    df = pd.read_parquet(f"{DATA_DIR}/txn_matrix_{split}.parquet")
    if n_tail: df = df.iloc[-n_tail:]
    y = df.pop("Is Laundering").to_numpy()
    return df.to_numpy("float32"), y

X_tr, y_tr = load("train", SUB)
X_va, y_va = load("val")
sc = StandardScaler().fit(X_tr)
X_tr_s, X_va_s = sc.transform(X_tr), sc.transform(X_va)
SPW = (y_tr == 0).sum() / y_tr.sum()

print(f"sweep-train {X_tr.shape} pos={y_tr.sum()} ({y_tr.mean()*100:.4f}%)")
print(f"val         {X_va.shape} pos={y_va.sum()} ({y_va.mean()*100:.4f}%)")
print(f"scale_pos_weight đầy đủ = {SPW:.0f}")

sweep-train (800000, 95) pos=813 (0.1016%)
val         (1015602, 95) pos=1082 (0.1065%)
scale_pos_weight day du = 983


In [2]:
SCALED = {"lr", "mlp"}

def build(name, p, seed=0):
    if name == "lr":  return LogisticRegression(**p, random_state=seed)
    if name == "dt":  return DecisionTreeClassifier(**p, random_state=seed)
    if name == "rf":  return RandomForestClassifier(**p, random_state=seed)
    if name == "mlp": return MLPClassifier(**p, random_state=seed)
    if name == "xgb": return XGBClassifier(**p, random_state=seed, eval_metric="aucpr")

rows = []
def run(name, params, factor, value, seed=0, Xa=None, ya=None):
    A, B = (X_tr_s, X_va_s) if name in SCALED else (X_tr, X_va)
    if Xa is not None: A = Xa
    yy = y_tr if ya is None else ya
    m = build(name, params, seed); t0 = time.time()
    m.fit(A, yy, eval_set=[(B, y_va)], verbose=False) if name == "xgb" else m.fit(A, yy)
    fit_s = time.time() - t0
    s = m.predict_proba(B)[:, 1]
    e = evaluate(y_va, s, find_best_threshold(y_va, s))
    nt = int(m.best_iteration) + 1 if name == "xgb" else None
    rows.append({"model": name, "factor": factor, "value": str(value), "seed": seed,
                 "f1": e["f1_minority"], "pr_auc": e["pr_auc"], "recall": e["recall"],
                 "precision": e["precision"], "fit_s": round(fit_s, 1), "n_trees": nt,
                 "params": json.dumps(params, default=str)})
    print(f"  {factor}={str(value):<18} f1={e['f1_minority']*100:5.2f} "
          f"pr_auc={e['pr_auc']*100:5.2f}"
          f"{'' if nt is None else f' cay={nt:>4}'} ({fit_s:.0f}s)", flush=True)

def boundary_table(rows, grids):
    """Kiem tra: toi uu co nam BEN TRONG khoang da quet khong.
       Cham bien => khoang quet (va do do GRIDS) qua hep."""
    ev = pd.DataFrame(rows); out = []
    for (mdl, fac), grid in grids.items():
        sub = ev[(ev.model == mdl) & (ev.factor == fac)]
        if sub.empty: continue
        vals = list(sub.value); best = sub.loc[sub.f1.idxmax(), "value"]
        out.append({"model": mdl, "sieu_tham_so": fac, "quet": vals, "toi_uu": best,
                    "co_trong_GRIDS": best in [str(g) for g in grid],
                    "khong_cham_bien": vals.index(best) not in (0, len(vals) - 1)})
    return pd.DataFrame(out)

In [3]:
print("--- dt: max_depth (GRIDS = [8,12,16]) ---")
for d in [4, 6, 8, 10, 12, 16, 20, None]:
    run("dt", {"max_depth": d, "class_weight": "balanced"}, "max_depth", d)

print("--- dt: class_weight ---")
for cw in ["balanced", None]:
    run("dt", {"max_depth": 10, "class_weight": cw}, "class_weight", cw)

--- dt: max_depth (GRIDS = [8,12,16]) ---
  max_depth=4                  f1= 8.76 pr_auc= 4.03 (13s)
  max_depth=6                  f1=53.57 pr_auc=33.63 (18s)
  max_depth=8                  f1=55.00 pr_auc=37.88 (27s)
  max_depth=10                 f1=54.60 pr_auc=37.17 (26s)
  max_depth=12                 f1=54.99 pr_auc=37.26 (27s)
  max_depth=16                 f1=49.29 pr_auc=28.03 (26s)
  max_depth=20                 f1=10.36 pr_auc= 2.04 (26s)
  max_depth=None               f1= 5.70 pr_auc= 0.52 (36s)
--- dt: class_weight ---
  class_weight=balanced           f1=54.60 pr_auc=37.17 (34s)
  class_weight=None               f1=17.22 pr_auc= 6.55 (32s)


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("results.csv")
metrics = ["f1_minority", "precision", "recall", "pr_auc",
           "recall@fpr1%", "precision@1000"]

agg = df.groupby(["model", "split"])[metrics].agg(["mean", "std"]).round(4)
agg

f1_minority         precision          recall          pr_auc  \
                   mean     std      mean     std    mean     std    mean   
model split                                                                 
dt    test       0.6215  0.0192    0.7237  0.0511  0.5457  0.0067  0.4359   
      val        0.5561  0.0168    0.7163  0.0496  0.4558  0.0163  0.3707   
lr    test       0.4291  0.0000    0.3730  0.0000  0.5050  0.0000  0.4622   
      val        0.3301  0.0000    0.3038  0.0000  0.3614  0.0000  0.2913   
mlp   test       0.7033  0.0115    0.7995  0.0394  0.6289  0.0195  0.7044   
      val        0.6313  0.0077    0.8263  0.0280  0.5111  0.0074  0.5956   
rf    test       0.6923  0.0016    0.8887  0.0077  0.5671  0.0049  0.6855   
      val        0.5308  0.1522    0.6794  0.2720  0.4495  0.0756  0.4942   
xgb   test       0.7386  0.0000    0.9203  0.0000  0.6168  0.0000  0.7298   
      val        0.6500  0.0213    0.8829  0.0467  0.5153  0.0229  0.6089   

                    recall@fpr1%         precision@1000          
                std         mean     std           mean     std  
model split                                                      
dt    test   0.0322       0.7761  0.0050         0.6480  0.0925  
      val    0.0300       0.7535  0.0269         0.5132  0.0217  
lr    test   0.0000       0.7764  0.0000         0.6190  0.0000  
      val    0.0000       0.7468  0.0000         0.3320  0.0000  
mlp   test   0.0129       0.8773  0.0079         0.9176  0.0235  
      val    0.0107       0.8257  0.0046         0.6006  0.0102  
rf    test   0.0010       0.8343  0.0041         0.9252  0.0016  
      val    0.1485       0.7978  0.0144         0.5129  0.1371  
xgb   test   0.0000       0.8804  0.0000         0.9500  0.0000  
      val    0.0311       0.8200  0.0146         0.6125  0.0256